In [ ]:
# plot_field_voltage.ipynb

# Cell 01 - Plot current vs. magnetic field strength

%matplotlib widget

import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures


def fit_linear(vec_x, vec_y):
    vec_x = vec_x.reshape(-1, 1)
    model = LinearRegression().fit(vec_x, vec_y)
    m = model.coef_
    b = model.intercept_
    score = model.score(vec_x, vec_y)
    return m, b, score


def fit_quintic(vec_x, vec_y):
    vec_x = vec_x.reshape(-1, 1)
    transformer = PolynomialFeatures(degree=5, include_bias=False)
    transformer.fit(vec_x)
    vec_x2 = transformer.transform(vec_x)
    model = LinearRegression().fit(vec_x2, vec_y)
    e, d, c, b, a = model.coef_
    f = model.intercept_
    score = model.score(vec_x2, vec_y)
    return a, b, c, d, e, f, score

# Create plot window with title and size
plt.close()
fig = plt.figure("Magnetic Field Strength")
fig.set_size_inches(11.5, 8)
ax = plt.gca()

# Read sample data file
file_name = "strength.csv"
folder_path = os.path.dirname(os.path.realpath("__file__"))
file_path = os.path.join(folder_path, file_name)
data = np.genfromtxt(f"{file_path}", delimiter=",")
vec_x = data[:, 0] / 256 * 5.0  # Convert to volts
vec_y = data[:, 1]
x = np.linspace(0, 5, 500)


# Plot the best linear fit
m, b, score = fit_linear(vec_x, vec_y)
ax.plot(
    x,
    m * x + b,
    color="green",
    linewidth=2,
    linestyle="--",
    label=f"Linear ($R^2$={score:.4f})",
)

# Plot the best quintic fit
a, b, c, d, e, f, score = fit_quintic(vec_x, vec_y)
ax.plot(
    x,
    a * x**5 + b * x**4 + c * x**3 + d * x**2 + e * x + f,
    color="blue",
    linewidth=2,
    label=f"Quintic ($R^2$={score:.4f})",
)

# Plot rising voltage strengths
ax.scatter(vec_x, vec_y, color="red", marker=".", label="Rising Volts")
# Plot falling voltage strengths
ax.scatter(vec_x[256:], vec_y[256:], color="purple", marker=".", label="Falling Volts")

# Decorate the graph with proper lables
ax.set_title(f"Magnetic Field Strength vs. Voltage")
ax.set_xlabel("Voltage (V)")
ax.set_ylabel("Field Strength (uT)")
ax.legend()
ax.xaxis.set_minor_locator(MultipleLocator(2))

# Save the graph as an image and Adobe PDF file
ax.figure.savefig(f"field_vs_voltage.png")
ax.figure.savefig(f"field_vs_voltage.pdf", dpi=600, orientation="landscape")

plt.show()